In [2]:
# @title Setup & Authentication
import getpass
import os

print("Please enter your VulnCheck API Key:")
# Using getpass ensures your API key is masked and not saved in the notebook's execution history
os.environ["VULNCHECK_TOKEN"] = getpass.getpass()
print("API Key registered securely in environment variables.")

Please enter your VulnCheck API Key:
··········
API Key registered securely in environment variables.


In [3]:
# @title Import Libraries & Define Target Assets
import requests
import json
import pandas as pd
from time import sleep
from google.colab import data_table
from google.colab import files

# Enable interactive, filterable tables natively in Colab
data_table.enable_dataframe_formatter()

BASE_URL = "https://api.vulncheck.com/v3"
HEADERS = {
    "accept": "application/json",
    "authorization": f"Bearer {os.environ.get('VULNCHECK_TOKEN')}"
}

# Acme Financial's Target Assets
cpe_list = [
    "cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:*:*:*:*:*",
    "cpe:2.3:a:smart-hmi:webiq:2.15.9:*:*:*:*:*:*:*",
    "cpe:2.3:a:ivanti:virtual_traffic_management:22.7:r1:*:*:*:*:*:*",
    "cpe:2.3:o:microsoft:windows_server_2025:10.0.26100.4946:*:*:*:*:*:x64:*"
]

In [4]:
# @title Define API Functions
def get_cves_for_cpe(cpe_string):
    print(f"[*] Querying CPE: {cpe_string}")
    url = f"{BASE_URL}/cpe?cpe={cpe_string}&isVulnerable=true"
    response = requests.get(url, headers=HEADERS)
    if response.status_code == 200:
        data = response.json()
        # The API returns a list of CVE strings directly in the 'data' field
        cves = data.get("data", [])
        print(f"    -> Found {len(cves)} CVEs.")
        return cves
    else:
        print(f"    -> Error querying CPE: {response.status_code}. Response text: {response.text}")
        return []


def enrich_cve(cve_id):
    """Single source of truth for CVE enrichment: CVSS/description/CISA-KEV
    from NVD2, plus exploit maturity, VulnCheck KEV, granular
    ransomware/botnet/APT attribution, timelines, and EPSS scores from the Exploits index."""
    enriched_data = {
        "CVE": cve_id,
        "CVSS V3 Base": None,
        "CISA KEV": False,
        "VulnCheck KEV": False,
        "Max Exploit Maturity": "unproven",
        "Ransomware Associated": False,
        "Botnet Associated": False,
        "APT Associated": False,
        "Threat Actors": "None",
        "NVD Published": None,
        "First Exploit Published": None,
        "Weaponized Exploit Published": None,
        "CISA KEV Date Added": None,
        "VulnCheck KEV Date Added": None,
        "EPSS Score": None,
        "EPSS Percentile": None,
        "Description": "N/A",
    }

    # 1. NVD2: CVSS, CISA KEV flag, description
    nvd_url = f"{BASE_URL}/index/vulncheck-nvd2?cve={cve_id}"
    nvd_resp = requests.get(nvd_url, headers=HEADERS)
    if nvd_resp.status_code == 200:
        nvd_data = nvd_resp.json().get("data", [])
        if nvd_data:
            vuln = nvd_data[0]
            enriched_data["Description"] = vuln.get("descriptions", [{}])[0].get("value", "N/A")
            enriched_data["CISA KEV"] = bool(vuln.get("cisaExploitAdd"))
            try:
                metrics = vuln.get("metrics", {})
                cvss_v3 = metrics.get("cvssMetricV31", metrics.get("cvssMetricV30", [{}]))[0]
                enriched_data["CVSS V3 Base"] = cvss_v3.get("cvssData", {}).get("baseScore")
            except (IndexError, KeyError):
                pass

    # 2. Exploits index: KEV status, maturity, Timelines, EPSS, and boolean attribution
    exploit_url = f"{BASE_URL}/index/exploits?cve={cve_id}"
    exploit_resp = requests.get(exploit_url, headers=HEADERS)
    if exploit_resp.status_code == 200:
        exploit_data = exploit_resp.json().get("data", [])
        if exploit_data:
            exp = exploit_data[0]
            enriched_data["Max Exploit Maturity"] = exp.get("max_exploit_maturity", "unproven")
            enriched_data["VulnCheck KEV"] = exp.get("inVCKEV", False)

            # --- Extract Exploit Timelines ---
            timeline = exp.get("timeline", {})
            enriched_data["NVD Published"] = timeline.get("nvd_published")
            enriched_data["First Exploit Published"] = timeline.get("first_exploit_published")
            enriched_data["Weaponized Exploit Published"] = timeline.get("first_exploit_published_weaponized_or_higher")
            enriched_data["CISA KEV Date Added"] = timeline.get("cisa_kev_date_added")
            enriched_data["VulnCheck KEV Date Added"] = timeline.get("vulncheck_kev_date_added")

            # --- Extract EPSS directly from Exploit Payload ---
            epss = exp.get("epss", {})
            enriched_data["EPSS Score"] = epss.get("epss_score")
            enriched_data["EPSS Percentile"] = epss.get("epss_percentile")

            # --- Use Top-Level Boolean Flags for Threat Activity ---
            enriched_data["Ransomware Associated"] = exp.get("reported_exploited_by_ransomware", False)
            enriched_data["Botnet Associated"] = exp.get("reported_exploited_by_botnets", False)
            enriched_data["APT Associated"] = exp.get("reported_exploited_by_threat_actors", False)

            # Still grab granular names if we want them for the "Threat Actors" string column
            ransomware_arr = exp.get("ransomware", []) or []
            botnet_arr = exp.get("botnets", []) or []
            threat_actor_arr = exp.get("threat_actors", []) or []

            all_actors = set()
            for arr in [ransomware_arr, botnet_arr, threat_actor_arr, exp.get("reported_exploitation", []) or []]:
                for item in arr:
                    if isinstance(item, dict):
                        name = item.get("name")
                        if name and name != "Unattributed" and "CVE-" not in name:
                            all_actors.add(name)

            if all_actors:
                enriched_data["Threat Actors"] = ", ".join(sorted(all_actors))

    return enriched_data

# Tier order mirrors the Evidence-Based Vulnerability Prioritization pyramid
PRIORITY_TIERS = [
    "Ransomware",
    "Botnets",
    "Threat Actors (APT)",
    "Unattributed KEV",
    "Weaponized",
    "Proof-of-Concept",
    "All Other Vulnerabilities",
]

def compute_priority_tier(row):
    """Assign each enriched CVE to a tier of the pyramid based on the
    granular attribution and exploit-maturity fields collected above."""
    if row["Ransomware Associated"]:
        return "Ransomware", "Associated with a known ransomware operation."
    if row["Botnet Associated"]:
        return "Botnets", "Associated with known botnet activity."
    if row["APT Associated"]:
        return "Threat Actors (APT)", f"Attributed to: {row['Threat Actors']}."
    if row["VulnCheck KEV"] or row["CISA KEV"]:
        return "Unattributed KEV", "Confirmed known-exploited; no specific actor/campaign attribution."
    maturity = (row["Max Exploit Maturity"] or "").lower()
    if maturity == "weaponized":
        return "Weaponized", "Weaponized exploit code available."
    if maturity in ("poc", "proof-of-concept", "proof of concept"):
        return "Proof-of-Concept", "Public proof-of-concept exists; not observed as weaponized or exploited."
    return "All Other Vulnerabilities", "No confirmed exploitation or public exploit code identified."


In [6]:
# @title Execute Pipeline & Build Dataset
print("Starting Vulnerability Enrichment Pipeline...\n")
all_results = []

for cpe in cpe_list:
    cves = get_cves_for_cpe(cpe)
    for i, cve in enumerate(cves):
        # Progress indicator
        if i % 5 == 0 and i > 0:
            print(f"    -> Enriched {i}/{len(cves)} CVEs...")

        enriched = enrich_cve(cve)
        enriched["Associated CPE"] = cpe
        all_results.append(enriched)

        # Respect API rate limits
        sleep(0.15)

print("\nPipeline Complete!\n")

# Convert the dictionary list into a Pandas DataFrame
df = pd.DataFrame(all_results)

# Priority tiering, aligned to the Evidence-Based Vulnerability Prioritization pyramid
tier_and_rationale = df.apply(compute_priority_tier, axis=1, result_type="expand")
df["Priority Tier"] = tier_and_rationale[0]
df["Priority Rationale"] = tier_and_rationale[1]

# Sort: highest-priority tier first, then by CVSS descending within tier
tier_rank = {t: i for i, t in enumerate(PRIORITY_TIERS)}
df["_tier_rank"] = df["Priority Tier"].map(tier_rank)
df = df.sort_values(
    by=["_tier_rank", "CVSS V3 Base"],
    ascending=[True, False],
).drop(columns="_tier_rank").reset_index(drop=True)

# Reorder columns for better readability
columns_order = [
    "CVE", "Associated CPE", "Priority Tier", "Priority Rationale",
    "CVSS V3 Base", "EPSS Score", "EPSS Percentile",
    "CISA KEV", "VulnCheck KEV", "Max Exploit Maturity",
    "NVD Published", "First Exploit Published", "Weaponized Exploit Published",
    "CISA KEV Date Added", "VulnCheck KEV Date Added",
    "Ransomware Associated", "Botnet Associated", "APT Associated",
    "Threat Actors", "Description",
]
df = df[columns_order]

print("\nPriority tier distribution:")
print(df["Priority Tier"].value_counts().reindex(PRIORITY_TIERS, fill_value=0))


Starting Vulnerability Enrichment Pipeline...

[*] Querying CPE: cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:*:*:*:*:*
    -> Found 38 CVEs.
    -> Enriched 5/38 CVEs...
    -> Enriched 10/38 CVEs...
    -> Enriched 15/38 CVEs...
    -> Enriched 20/38 CVEs...
    -> Enriched 25/38 CVEs...
    -> Enriched 30/38 CVEs...
    -> Enriched 35/38 CVEs...
[*] Querying CPE: cpe:2.3:a:smart-hmi:webiq:2.15.9:*:*:*:*:*:*:*
    -> Found 1 CVEs.
[*] Querying CPE: cpe:2.3:a:ivanti:virtual_traffic_management:22.7:r1:*:*:*:*:*:*
    -> Found 0 CVEs.
[*] Querying CPE: cpe:2.3:o:microsoft:windows_server_2025:10.0.26100.4946:*:*:*:*:*:x64:*
    -> Found 1293 CVEs.
    -> Enriched 5/1293 CVEs...
    -> Enriched 10/1293 CVEs...
    -> Enriched 15/1293 CVEs...
    -> Enriched 20/1293 CVEs...
    -> Enriched 25/1293 CVEs...
    -> Enriched 30/1293 CVEs...
    -> Enriched 35/1293 CVEs...
    -> Enriched 40/1293 CVEs...
    -> Enriched 45/1293 CVEs...
    -> Enriched 50/1293 CVEs...
    -> Enriched 55/1293 CV

In [7]:
# @title View Results & Download CSV

# 1. Display as an interactive, filterable table directly in Colab
display(df)

# 2. Save to CSV
csv_filename = "acme_enriched_cves.csv"
df.to_csv(csv_filename, index=False)
print(f"\nSaved results to {csv_filename}")

# 3. Also save JSON (useful for feeding the Step 2 dashboard directly)
json_filename = "acme_enriched_cves.json"
df.to_json(json_filename, orient="records", indent=2)
print(f"Saved results to {json_filename}")

# 4. Trigger file download to your local machine
files.download(csv_filename)
files.download(json_filename)


,CVE,Associated CPE,Priority Tier,Priority Rationale,CVSS V3 Base,EPSS Score,EPSS Percentile,CISA KEV,VulnCheck KEV,Max Exploit Maturity,NVD Published,First Exploit Published,Weaponized Exploit Published,CISA KEV Date Added,VulnCheck KEV Date Added,Ransomware Associated,Botnet Associated,APT Associated,Threat Actors,Description
0,CVE-2024-0012,cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:...,Ransomware,Associated with a known ransomware operation.,9.8,0.99698,0.99951,True,True,weaponized,2024-11-18T16:15:11.683Z,2024-11-19T00:00:00Z,2024-11-19T00:00:00Z,2024-11-18T00:00:00Z,2024-11-18T00:00:00Z,True,True,True,"China Attribution, Linuxsys, MgBot, Palo Alto ...",An authentication bypass in Palo Alto Networks...
1,CVE-2026-0300,cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:...,Ransomware,Associated with a known ransomware operation.,9.8,0.32074,0.98166,True,True,weaponized,2026-05-06T19:16:35.73Z,None,None,2026-05-06T00:00:00Z,2026-05-05T00:00:00Z,True,False,True,"China Attribution, Palo Alto Networks PAN-OS O...",A buffer overflow vulnerability in the User-ID...
2,CVE-2025-59287,cpe:2.3:o:microsoft:windows_server_2025:10.0.2...,Ransomware,Associated with a known ransomware operation.,9.8,0.99938,0.99971,True,True,weaponized,2025-10-14T17:16:11.67Z,2025-10-17T00:00:00Z,2025-10-21T00:00:00Z,2025-10-24T00:00:00Z,2025-10-24T00:00:00Z,True,False,True,"China Attribution, Iran Attribution, Microsoft...",Deserialization of untrusted data in Windows S...
3,CVE-2026-0257,cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:...,Ransomware,Associated with a known ransomware operation.,9.1,0.93905,0.99837,True,True,weaponized,2026-05-13T19:17:01.04Z,2026-05-29T00:00:00Z,2026-06-01T00:00:00Z,2026-05-29T00:00:00Z,2026-05-29T00:00:00Z,True,False,True,Palo Alto Networks PAN-OS Authentication Bypas...,Authentication bypass vulnerabilities in the G...
4,CVE-2025-60710,cpe:2.3:o:microsoft:windows_server_2025:10.0.2...,Ransomware,Associated with a known ransomware operation.,7.8,0.04598,0.90869,True,True,weaponized,2025-11-11T18:15:39.073Z,2025-11-11T00:00:00Z,None,2026-04-13T00:00:00Z,2026-04-13T00:00:00Z,True,False,False,Microsoft Windows Link Following Vulnerability,Improper link resolution before file access ('...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1327,CVE-2025-0116,cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:...,All Other Vulnerabilities,No confirmed exploitation or public exploit co...,NaN,NaN,NaN,False,False,unproven,None,None,None,None,None,False,False,False,None,A Denial of Service (DoS) vulnerability in Pal...
1328,CVE-2025-0123,cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:...,All Other Vulnerabilities,No confirmed exploitation or public exploit co...,NaN,NaN,NaN,False,False,unproven,None,None,None,None,None,False,False,False,None,A vulnerability in the Palo Alto Networks PAN-...
1329,CVE-2025-0125,cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:...,All Other Vulnerabilities,No confirmed exploitation or public exploit co...,NaN,NaN,NaN,False,False,unproven,None,None,None,None,None,False,False,False,None,An improper input neutralization vulnerability...
1330,CVE-2025-0137,cpe:2.3:o:paloaltonetworks:pan-os:11.2.4:h2:*:...,All Other Vulnerabilities,No confirmed exploitation or public exploit co...,NaN,NaN,NaN,False,False,unproven,None,None,None,None,None,False,False,False,None,An improper input neutralization vulnerability...



Saved results to acme_enriched_cves.csv
Saved results to acme_enriched_cves.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>